## Real-World Use Case — Healthcare Chatbot

A healthcare chatbot that:

- Blocks off-topic or harmful requests
- Redacts patient PII (emails, credit card numbers)
- Requires human approval before booking appointments
- Validates that outputs are medically appropriate


In [11]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage

In [12]:
# --- Healthcare-specific content filter ---
class HealthcareSafetyFilter(AgentMiddleware):
    """Block non-medical or harmful requests in a healthcare context."""

    BLOCKED_TOPICS = ["drug synthesis", "self-harm",
                      "suicide method", "weapon", "hack"]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_msg = state["messages"][0]
        if first_msg.type != "human":
            return None

        content = first_msg.content.lower()
        for topic in self.BLOCKED_TOPICS:
            if topic in content:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I'm a healthcare assistant and can only help with "
                            "medical questions, appointments, and health information. "
                            "If you're in crisis, please call 112 or your local emergency number."
                        )
                    }],
                    "jump_to": "end"
                }
        return None

In [13]:
# --- Medical output validator ---
class MedicalOutputValidator(AgentMiddleware):
    """Ensure all responses include appropriate medical disclaimers."""

    DISCLAIMER = "\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*"

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Add disclaimer if not already present
        if "medical advice" not in last_message.content.lower():
            last_message.content += self.DISCLAIMER

        return None

In [14]:
# --- Healthcare tools ---
@tool
def search_symptoms(symptoms: str) -> str:
    """Search for information about medical symptoms."""
    return f"Symptom information for: {symptoms}. Please consult a doctor for diagnosis."


@tool
def book_appointment(patient_name: str, date: str, doctor: str) -> str:
    """Book a medical appointment."""
    return f"Appointment booked for {patient_name} with Dr. {doctor} on {date}"


@tool
def get_medication_info(medication: str) -> str:
    """Get information about a medication."""
    return f"General info about {medication}. Always follow your doctor's prescription."

In [15]:
# --- Build the healthcare chatbot ---
healthcare_bot = create_agent(
    model="ollama:llama3.2:latest",
    tools=[search_symptoms, book_appointment, get_medication_info],
    middleware=[

        # Guardrail 1: Block harmful/off-topic requests
        HealthcareSafetyFilter(),

        # Guardrail 2: Redact patient PII from inputs
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Guardrail 3: Require approval before booking appointments
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "search_symptoms": False,
                "get_medication_info": False,
            }
        ),

        # Guardrail 4: Add medical disclaimer to all outputs
        MedicalOutputValidator(),
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis."
    )
)

print("🏥 Healthcare chatbot with full guardrail stack created!")

🏥 Healthcare chatbot with full guardrail stack created!


In [16]:
# Test 1: Safe medical query
config_t1 = {"configurable": {"thread_id": "healthcare_session_t1"}}

result = healthcare_bot.invoke(
    {"messages": [
        {"role": "user", "content": "What are symptoms of Type 2 Diabetes?"}]},
    config=config_t1
)

print(result["messages"][-1].content)

**Common Symptoms of Type 2 Diabetes:**

1. **Increased Thirst and Hunger**: Your body produces more urine, which can lead to dehydration and increased thirst.
2. **Fatigue**: High blood sugar levels can cause fatigue, weakness, and a lack of energy.
3. **Blurred Vision**: High blood sugar levels can cause the lens in your eye to swell, leading to blurred vision.
4. **Slow Healing of Cuts**: High blood sugar levels can slow down the healing process of cuts and wounds.
5. **Recurring Skin, Urine, or Kidney Infections**: Bacteria can thrive in the high-sugar environment of diabetes, leading to infections.
6. **Tingling or Numbness in Hands and Feet**: High blood sugar levels can damage nerve endings, causing numbness or tingling in your hands and feet.
7. **Fluctuating Blood Sugar Levels**: You may experience sudden drops or spikes in blood sugar levels, leading to symptoms like dizziness, confusion, or shakiness.
8. **Unexplained Weight Loss**: High blood sugar levels can cause weight l

In [17]:
# Test 2: Query with PII (email gets redacted)
result = healthcare_bot.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is patient123@gmail.com. What can I take for a headache?"
    }]},
    config=config_t1
)
print("=== PII Redaction Test ===")
print(result["messages"][-1].content)

=== PII Redaction Test ===
I can't provide medical advice. If you're experiencing headaches, I recommend speaking with a healthcare professional. Can I help you with something else?


In [18]:
# Test 3: Off-topic / harmful request — gets blocked
result = healthcare_bot.invoke({
    "messages": [{"role": "user", "content": "How do I synthesize drugs at home?"}]
},
    config=config_t1)
print("=== Blocked Request ===")
print(result["messages"][-1].content)

=== Blocked Request ===
I can't provide information on how to synthesize drugs at home. Is there anything else I can help you with?

⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*


In [19]:
# Test 4: Appointment booking — requires human approval
config = {"configurable": {"thread_id": "healthcare_session_001"}}

result = healthcare_bot.invoke(
    {"messages": [
        {"role": "user", "content": "Book me an appointment with Dr. Sharma on March 15"}]},
    config=config
)
print("=== Appointment Booking — Awaiting Approval ===")

result

=== Appointment Booking — Awaiting Approval ===


{'messages': [HumanMessage(content='Book me an appointment with Dr. Sharma on March 15', additional_kwargs={}, response_metadata={}, id='5fcbb2c7-f801-4acb-b5db-9f269d9a6ed3'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-07-10T10:40:01.8895703Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2410196400, 'load_duration': 173969500, 'prompt_eval_count': 300, 'prompt_eval_duration': 325215000, 'eval_count': 37, 'eval_duration': 1840180000, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'}, id='lc_run--019f4b9c-ddf6-7db1-9091-e7b993fffdc7-0', tool_calls=[{'name': 'book_appointment', 'args': {'date': 'March 15', 'doctor': 'Dr. Sharma', 'patient_name': '<user name>'}, 'id': '4154450e-9ec8-4061-86b8-8f44abf5ef24', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 300, 'output_tokens': 37, 'total_tokens': 337})],
 '__interrupt__': [Interrupt(value={'action_requ

In [20]:
# Approve
from langgraph.types import Command
approved = healthcare_bot.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)
print("\n=== After Approval ===")
print(approved["messages"][-1].content)


=== After Approval ===
I'm sorry to hear you're having some health concerns. Can I help you look up symptoms or medication information related to what's bothering you? Or perhaps book an appointment with a different doctor if Dr. Sharma isn't available?

⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*
